In [1]:
import functools
import operator

import yaml

In [2]:
config = "longcrawl_1176m"
tokens = 1e10

In [3]:
vocab = seq_len = layers = d_model = n_q_per_kv = n_kv = d_head = d_ff = 0
nsa_l = nsa_d = nsa_L = nsa_n = nsa_w = 0
with open(f"configs/{config}.yaml", "r") as f:
    config = yaml.safe_load(f)
locals().update(config["model"])
batch_size = config["optimizer"]["batch_size"]

In [4]:
parameter_shapes = {
    "embed": (vocab, d_model),
    "ln_embed": (d_model,),
    "ln_final": (d_model,),
    "transformerln_attn_in": (layers, d_model),
    "transformerln_attn_out": (layers, d_model),
    "transformerln_ffn_in": (layers, d_model),
    "transformerln_ffn_out": (layers, d_model),
    "transformerln_k": (layers, 3, n_kv, d_head),
    "transformerln_q": (layers, n_q_per_kv, n_kv, d_head),
    "transformerln_qkv": (layers, n_q_per_kv, n_kv, d_head),
    "transformerphi": (layers, nsa_l * d_head, n_kv, d_head),
    "transformerk_intrablock_pe": (layers, nsa_l, n_kv, d_head),
    "transformerv_intrablock_pe": (layers, nsa_l, n_kv, d_head),
    "transformerw_down": (layers, d_model, d_ff),
    "transformerw_gate": (layers, d_model, d_ff),
    "transformerw_k": (layers, 3, d_model, n_kv, d_head),
    "transformerw_nsa_gate": (layers, d_head, n_q_per_kv, n_kv, 3),
    "transformerw_o": (layers, d_model, n_q_per_kv, n_kv, d_head),
    "transformerw_q": (layers, d_model, n_q_per_kv, n_kv, d_head),
    "transformerw_up": (layers, d_model, d_ff),
    "transformerw_v": (layers, 3, d_model, n_kv, d_head),
    "unembed": (vocab, d_model),
}

In [5]:
N_nonembedding = sum(
    {k: functools.reduce(operator.mul, v, 1) for k, v in parameter_shapes.items() if k != "embed"}.values()
)
print(f"N_nonembedding: {N_nonembedding:,}")

N_nonembedding: 1,176,289,280


In [6]:
def round_to_nearest_10(x):
    return round(x / 10) * 10


print(f"optimizer.steps: {round_to_nearest_10(tokens / (batch_size * seq_len))}")

optimizer.steps: 19070
